In [43]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

from sklearn.preprocessing import LabelEncoder

from xgboost import XGBClassifier

import joblib

In [44]:
df = pd.read_csv(
    "../data/raw/accepted_2007_to_2018Q4.csv.gz",
    nrows=100000,
    low_memory=False
)

print(df.shape)

(100000, 151)


In [45]:
df = df[
    df["loan_status"].isin([
        "Fully Paid",
        "Charged Off"
    ])
]

df["target"] = df["loan_status"].map({
    "Fully Paid": 0,
    "Charged Off": 1
})

df["fico_score"] = (
    df["fico_range_low"] +
    df["fico_range_high"]
) / 2

df["earliest_cr_line"] = pd.to_datetime(
    df["earliest_cr_line"],
    format="%b-%Y",
    errors="coerce"
)

df["credit_history_years"] = (
    pd.Timestamp("2019-01-01") -
    df["earliest_cr_line"]
).dt.days / 365

df["loan_income_ratio"] = (
    df["loan_amnt"] /
    (df["annual_inc"] + 1)
)

df["balance_income_ratio"] = (
    df["tot_cur_bal"] /
    (df["annual_inc"] + 1)
)

df["accounts_per_year"] = (
    df["total_acc"] /
    (df["credit_history_years"] + 1)
)

df["credit_utilization_score"] = (
    df["revol_util"] *
    df["bc_util"]
)
print(df["target"].value_counts())

target
0    70288
1    17603
Name: count, dtype: int64


C:\Windows\Temp\ipykernel_17248\548219865.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["target"] = df["loan_status"].map({
C:\Windows\Temp\ipykernel_17248\548219865.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["fico_score"] = (
C:\Windows\Temp\ipykernel_17248\548219865.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmen

In [46]:
FEATURES = [
    "inq_last_6mths",
    "sub_grade",
    "acc_open_past_24mths",
    "grade",
    "delinq_2yrs",
    "fico_score",
    "pct_tl_nvr_dlq",
    "emp_length",
    "verification_status",
    "loan_income_ratio",
    "loan_amnt",
    "open_acc",
    "purpose",
    "installment",
    "total_acc"
]

In [47]:
df_model = df[FEATURES + ["target"]].copy()

df_model = df_model.replace(
    [np.inf, -np.inf],
    np.nan
)

for col in df_model.columns:

    if col == "target":
        continue

    if pd.api.types.is_numeric_dtype(df_model[col]):

        df_model[col] = (
            df_model[col]
            .fillna(df_model[col].median())
        )

    else:

        df_model[col] = (
            df_model[col]
            .fillna(df_model[col].mode()[0])
        )

In [48]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = [
    "grade",
    "sub_grade",
    "purpose",
    "verification_status"
]

encoders = {}

for col in categorical_cols:

    le = LabelEncoder()

    df_model[col] = le.fit_transform(
        df_model[col].astype(str)
    )

    encoders[col] = le

In [49]:
df_model["emp_length"] = (
    df_model["emp_length"]
    .astype(str)
    .str.replace("years", "", regex=False)
    .str.replace("year", "", regex=False)
    .str.replace("+", "", regex=False)
    .str.replace("< 1", "0", regex=False)
    .str.strip()
)

df_model["emp_length"] = pd.to_numeric(
    df_model["emp_length"],
    errors="coerce"
)

df_model["emp_length"] = (
    df_model["emp_length"]
    .fillna(df_model["emp_length"].median())
)
print(df_model["emp_length"].head())
print(df_model["emp_length"].dtype)
print(X.dtypes[X.dtypes == "object"])

0    10
1    10
2    10
4     3
5     4
Name: emp_length, dtype: int64
int64
Series([], dtype: object)


In [50]:
print(df_model.dtypes)
print(df_model.isnull().sum().sum())

inq_last_6mths          float64
sub_grade                 int64
acc_open_past_24mths    float64
grade                     int64
delinq_2yrs             float64
fico_score              float64
pct_tl_nvr_dlq          float64
emp_length                int64
verification_status       int64
loan_income_ratio       float64
loan_amnt               float64
open_acc                float64
purpose                   int64
installment             float64
total_acc               float64
target                    int64
dtype: object
0


In [51]:
# np.random.seed(42)

# df_model["psychometric_score"] = np.random.randint(
#     50,
#     100,
#     len(df_model)
# )

# df_model["phone_bill_score"] = np.random.randint(
#     50,
#     100,
#     len(df_model)
# )

# df_model["ecommerce_score"] = np.random.randint(
#     40,
#     100,
#     len(df_model)
# )

In [52]:
X = df_model.drop("target", axis=1)

y = df_model["target"]

In [53]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

negative = (y_train == 0).sum()
positive = (y_train == 1).sum()

scale_weight = negative / positive

print("Negative:", negative)
print("Positive:", positive)
print("Scale Weight:", scale_weight)

Negative: 56230
Positive: 14082
Scale Weight: 3.9930407612555037


In [ ]:
# best_model = XGBClassifier(
#     n_estimators=500,
#     max_depth=6,
#     learning_rate=0.05,
#     min_child_weight=3,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     scale_pos_weight=1,
#     random_state=42,
#     eval_metric="logloss"
# )
 
# best_model.fit(
#     X_train_smote,
#     y_train_smote
# )

In [55]:
from sklearn.model_selection import RandomizedSearchCV

params = {
    "max_depth": [4, 6, 8, 10],
    "learning_rate": [0.01, 0.03, 0.05],
    "n_estimators": [300, 500, 800],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

search = RandomizedSearchCV(
    estimator=XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    ),
    param_distributions=params,
    n_iter=15,
    scoring="roc_auc",
    cv=3,
    verbose=2,
    n_jobs=-1
)

search.fit(
    X_train_smote,
    y_train_smote
)

print(search.best_params_)
print(search.best_score_)

Fitting 3 folds for each of 15 candidates, totalling 45 fits
{'subsample': 0.8, 'n_estimators': 800, 'max_depth': 10, 'learning_rate': 0.01, 'colsample_bytree': 1.0}
0.9150365052154613


In [62]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Probabilities
probs = best_model.predict_proba(X_test)[:, 1]

# Best threshold found earlier
threshold = 0.26

# Convert probabilities to predictions
pred = (probs > threshold).astype(int)

print("="*50)
print(f"Threshold : {threshold}")
print("="*50)

print("Accuracy :", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred))
print("Recall   :", recall_score(y_test, pred))
print("F1 Score :", f1_score(y_test, pred))
print("ROC AUC  :", roc_auc_score(y_test, probs))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, pred))

Threshold : 0.26
Accuracy : 0.7144319927185847
Precision: 0.35935447551135297
Recall   : 0.5438795796648679
F1 Score : 0.4327683615819209
ROC AUC  : 0.7236154845008764

Confusion Matrix
[[10644  3414]
 [ 1606  1915]]


In [57]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

probs = best_model.predict_proba(X_test)[:,1]

best_threshold = 0
best_f1 = 0

for threshold in np.arange(0.1, 0.9, 0.01):

    pred = (probs > threshold).astype(int)

    f1 = f1_score(y_test, pred)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

print("Best Threshold:", best_threshold)
print("Best F1:", best_f1)


Best Threshold: 0.18999999999999995
Best F1: 0.43646029840507633


In [58]:
from sklearn.metrics import roc_auc_score

probs = best_model.predict_proba(X_test)[:,1]

print(
    "ROC AUC:",
    roc_auc_score(y_test, probs)
)

ROC AUC: 0.7236154845008764


In [59]:
importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": best_model.feature_importances_
})

importance_df.sort_values(
    "importance",
    ascending=False
)

,feature,importance
0,inq_last_6mths,0.321206
2,acc_open_past_24mths,0.146277
3,grade,0.130063
4,delinq_2yrs,0.108670
1,sub_grade,0.092257
5,fico_score,0.042008
7,emp_length,0.027838
11,open_acc,0.024246
6,pct_tl_nvr_dlq,0.023937
8,verification_status,0.019603


In [60]:
import joblib

import joblib

joblib.dump(best_model, "../models/model_v2.pkl")
joblib.dump(encoders, "../models/encoders2.pkl")

['../models/encoders2.pkl']

In [61]:
y.value_counts(normalize=True)

target
0    0.799718
1    0.200282
Name: proportion, dtype: float64